# LAB-HW-10 — AXI / Burst Measurement

**One new thing today:** make programmable logic move real DDR data through AXI, then measure how transaction granularity changes an end-to-end DMA workload.

Prerequisites: LSN-018, LAB-HW-09 integrity PASS, and the established PS/Linux + JTAG path.

**Project Trace:** RMD-014A · T-HW-010/T-HW-011

You will use AMD AXI Central Direct Memory Access (AXI CDMA). You do **not** write a full AXI master from scratch.

## 1. What changed since HW-09?

HW-09 proved that large system memory can preserve known bytes from PS/Linux.

HW-10 changes the data mover:

```console
HW-09: CPU / Linux memory operations → DDR integrity

HW-10: PL AXI CDMA → S_AXI_HP0_FPD → DDR
```

The comparison is now about a **real PL AXI master path**.

It is still not a formal FlyBrain DDR-backed synapse store.

## 2. The physical path

<svg xmlns="http://www.w3.org/2000/svg" width="1040" height="390" viewBox="0 0 1040 390" role="img" aria-label="LAB-HW-10 AXI CDMA control and DDR data paths">
  <rect x="25" y="70" width="155" height="80" rx="10" fill="#eef0ff" stroke="#333"/>
  <text x="102" y="103" text-anchor="middle" font-size="14">PS / Linux</text>
  <text x="102" y="127" text-anchor="middle" font-size="12">benchmark.py</text>
  <rect x="230" y="45" width="185" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="322" y="78" text-anchor="middle" font-size="14">M_AXI_HPM0_FPD</text>
  <text x="322" y="102" text-anchor="middle" font-size="12">control path</text>
  <rect x="470" y="45" width="180" height="80" rx="10" fill="#fff3cd" stroke="#333"/>
  <text x="560" y="78" text-anchor="middle" font-size="14">AXI CDMA</text>
  <text x="560" y="102" text-anchor="middle" font-size="12">S_AXI_LITE 0xA0020000</text>
  <rect x="470" y="220" width="180" height="80" rx="10" fill="#fff3cd" stroke="#333"/>
  <text x="560" y="253" text-anchor="middle" font-size="14">AXI CDMA M_AXI</text>
  <text x="560" y="277" text-anchor="middle" font-size="12">128 bit / burst ≤64</text>
  <rect x="700" y="220" width="150" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="775" y="253" text-anchor="middle" font-size="14">S_AXI_HP0_FPD</text>
  <text x="775" y="277" text-anchor="middle" font-size="12">non-coherent</text>
  <rect x="895" y="220" width="120" height="80" rx="10" fill="#fee" stroke="#333"/>
  <text x="955" y="253" text-anchor="middle" font-size="14">DDR4</text>
  <text x="955" y="277" text-anchor="middle" font-size="12">DMA buffer</text>
  <path d="M180 100 L230 85 M415 85 L470 85 M560 125 L560 220 M650 260 L700 260 M850 260 L895 260" stroke="#333" stroke-width="2" fill="none"/>
</svg>

Control and data are different paths:

- PS writes AXI CDMA registers through `M_AXI_HPM0_FPD`.
- CDMA becomes the AXI master for the DDR copy through `S_AXI_HP0_FPD`.

## 3. Why the DMA buffer needs a special contract

AMD documents `S_AXI_HP0_FPD` as a **non-coherent** PL-to-DDR path.

So ordinary cached Python memory is not a safe teaching assumption for DMA.

Physical mode requires:

- `/dev/udmabuf0`;
- at least **2 MiB**;
- physical address exposed by `/sys/class/u-dma-buf/udmabuf0/phys_addr`;
- the helper opens the device with `O_SYNC`;
- the whole course window must lie inside `HP0_DDR_LOW`.

The course uses the upstream **u-dma-buf** project as a buffer provider. You are not asked to write or modify a kernel driver.

If these conditions are not available on the selected Ubuntu/kernel image, stop and save the failure evidence. Do not substitute a guessed DDR physical address.

## 4. Same bytes, different transaction granularity

Both patterns copy the exact same **256 KiB** payload from the same source region to the same destination region.

### Contiguous

```console
1 request × 256 KiB
```

### Small/scattered

```console
1024 requests × 256 B
```

Block order is deterministic:

```text
block = (257*i + 17) mod 1024
```

Because 257 is coprime with 1024, every block appears exactly once.

No random seed is needed.

## 5. What exactly is timed?

The timer starts before the helper programs the first transfer and stops after the final DMA completion poll.

Therefore timing includes:

- Python register writes;
- Python status polling;
- AXI CDMA command overhead;
- AXI transfer;
- DDR path.

It excludes payload generation and the post-run byte comparison.

This is an **end-to-end software-controlled DMA workload**, not a pure AXI bus analyzer measurement.

## 6. Dry-run the workload and statistics first

```bash
python boards/kv260/runtime/axi_cdma_benchmark.py \
  --dry-run \
  --json-out /tmp/lab-hw-10-dry-run.json
```

The dry-run executes the same block ordering and integrity logic against a software memory model. Timing samples are deterministic synthetic values so CI does not become a noisy performance test.

Expected markers include:

```console
PRECHECK_INTEGRITY=PASS
MEASUREMENT_STABLE=1
PERFORMANCE_CONCLUSION_ALLOWED=1
STATUS=PASS
```

A CI PASS is not a physical bandwidth result.

## 7. Prove both failure gates

Integrity failure:

```bash
python boards/kv260/runtime/axi_cdma_benchmark.py \
  --dry-run --inject-corruption-for-test
```

Expected: `BENCHMARK_INTEGRITY_MISMATCH`, no performance conclusion.

Stability failure:

```bash
python boards/kv260/runtime/axi_cdma_benchmark.py \
  --dry-run --unstable-for-test
```

Expected: `MEASUREMENT_UNSTABLE` and `PERFORMANCE_CONCLUSION_ALLOWED=0`.

A benchmark that cannot fail these two tests is not ready for a real board.

## 8. Build the AXI CDMA bitstream

Development host:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-10-build.log \
  -source boards/kv260/scripts/build_lab10_axi_cdma.tcl
```

Frozen build facts:

- Simple DMA;
- Scatter/Gather off;
- 128-bit M_AXI;
- max burst 64;
- 64-bit address registers;
- `S_AXI_HP0_FPD`;
- control base `0xA0020000`;
- only `HP0_DDR_LOW` mapped for the first Lab;
- bitstream is blocked by DRC or negative setup/hold slack.

Expected bitstream:

`build/kv260/lab-hw-10/kv260_axi_cdma_benchmark.bit`

## 9. Program without silently reinitializing the running PS

Keep the LAB-HW-05 Linux system running. If an active Kria application is loaded, unload it first.

Program the PL with the shared direct-JTAG helper:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-10-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-10/kv260_axi_cdma_benchmark.bit
```

Do **not** run a PS initialization sequence that resets/reconfigures the live Linux DDR subsystem merely to force the Lab to work.

If the selected boot firmware does not leave the required HP0 runtime path usable, save the failure and keep T-HW-010 blocked; revise the platform flow later.

## 10. Check the DMA-safe buffer provider

Before the physical benchmark, the runtime host must provide u-dma-buf.

Check:

```bash
ls -l /dev/udmabuf0
cat /sys/class/u-dma-buf/udmabuf0/phys_addr
cat /sys/class/u-dma-buf/udmabuf0/size
```

Required size: at least **2097152 bytes**.

If the device is absent, prepare the course image using the upstream u-dma-buf project that matches the running kernel. The upstream driver can allocate a userspace-mappable contiguous DMA buffer and exposes its physical address in sysfs.

Do not download/build a random kernel module during a graded run without recording exactly what version was installed.

## 11. Run the real benchmark

Copy `axi_cdma_benchmark.py` to PS/Linux and run:

```bash
sudo python3 /tmp/axi_cdma_benchmark.py \
  --physical \
  --json-out /tmp/lab-hw-10-trace.json
```

Physical mode requires root because the AXI CDMA control registers are accessed through fixed `/dev/mem` MMIO.

The helper rejects:

- missing u-dma-buf;
- buffer smaller than 2 MiB;
- buffer outside `HP0_DDR_LOW`;
- CDMA reset/timeout/error;
- data mismatch;
- unstable two-batch measurement.

A PASS requires both patterns to satisfy integrity and the ≤10% stability gate.

## 12. Read the result without overclaiming

For each pattern and each batch the checker records all 20 raw samples plus:

- median;
- minimum;
- maximum;
- median MiB/s.

Then it reports the relative difference between batch medians.

Only when both patterns are stable does it print:

`SCATTERED_TO_CONTIGUOUS_MEDIAN_TIME_RATIO=...`

Interpret that ratio only as:

> under this exact bitstream, 256 KiB workload, Python polling boundary, AXI CDMA configuration, and runtime session, the two transaction patterns produced these observed end-to-end times.

Do not rewrite it as “DDR is N× faster” or “AXI bursts are always N× faster.”

## 13. Failure classes

Important classes include:

- `DMA_BUFFER_DEVICE_MISSING`
- `DMA_BUFFER_SYSFS_MISSING`
- `DMA_BUFFER_TOO_SMALL`
- `DMA_BUFFER_OUTSIDE_HP0_DDR_LOW`
- `TRANSPORT_REQUIRES_ROOT`
- `TRANSPORT_PERMISSION_OR_POLICY`
- `CDMA_RESET_TIMEOUT`
- `CDMA_TIMEOUT`
- `CDMA_ERROR`
- `BENCHMARK_INTEGRITY_MISMATCH`
- `POST_BENCHMARK_INTEGRITY_MISMATCH`
- `MEASUREMENT_UNSTABLE`

Transport/platform failures, correctness failures, and measurement-quality failures are different engineering problems.

## 14. Expected Evidence / Save Evidence

Retain:

- Git commit;
- board/carrier revision;
- Ubuntu/kernel identity;
- u-dma-buf source/version identity used by the course image;
- `/dev/udmabuf0` size and physical base;
- buffer cache-mode contract (`O_SYNC`);
- LAB-HW-10 bitstream SHA-256;
- Vivado build, DRC, timing, utilization, and program logs;
- AXI CDMA configuration and control base;
- helper SHA-256;
- payload SHA-256;
- precheck and post-batch integrity results;
- every warm-up count and every measured raw sample;
- two medians/min/max per pattern;
- stability percentages;
- ratio only when conclusion is allowed;
- `lab-hw-10-trace.json`;
- experiment date.

This closes the Physical-Lab teaching track at the CI/documentation level, but real T-HW-010 still requires real-board evidence.

## 15. Human Check

Explain:

1. Why are control AXI and DDR data AXI different paths?
2. Why is `S_AXI_HP0_FPD` called non-coherent?
3. Why can ordinary cached Python memory not be assumed DMA-safe?
4. Why do both patterns move the same 256 KiB?
5. Why does small/scattered use a deterministic permutation?
6. What does the timer include?
7. Why are 5 warm-ups excluded?
8. Why use median rather than one “best” sample?
9. Why does >10% batch drift block the conclusion?
10. Why is the final ratio not a peak-DDR claim?

## 16. Official / implementation basis

AMD sources:

- PG201: `S_AXI_HP0_FPD` is a non-coherent PL-to-FPD/DDR path; the HP interfaces support up to 128-bit data width.
- PG034: AXI CDMA provides a memory-mapped AXI4 master plus AXI4-Lite control; Simple DMA is supported; the DataMover automatically partitions bursts and protects 4 KiB boundaries.
- PG034 register map: `CDMACR=0x00`, `CDMASR=0x04`, `SA=0x18`, `SA_MSB=0x1C`, `DA=0x20`, `DA_MSB=0x24`, `BTT=0x28`.
- PG034 throughput guidance: larger configured burst lengths can increase realized CDMA bus utilization, but system-level performance still depends on the workload.

Buffer-provider basis:

- upstream u-dma-buf documents a userspace-mappable contiguous DMA buffer, sysfs physical-address exposure, Zynq UltraScale+ ARM64 support, and cache behavior controlled by `O_SYNC`.

This Lab freezes one teaching configuration; it does not make u-dma-buf or AXI CDMA a permanent FlyBrain architecture choice.